CUSTOM LAYERS IN TENSORFLOW: OLTRE I CONFINI DI KERAS

Sappiamo costruire i nostri modelli con i mattoncini che Keras mette a disposizione.
ma cosa succede se dobbiamo utilizzare un mattoncino non disponibile nel catalogo di Keras?

- Estensione della classe base (basi dell'ereditarietà)
- Gestione del ciclo di vita del layer: implementazione di 'build' e 'call'
- Sviluppo pratico: scrivere un layer che applica una trasformazione matematica personalizzata.

Keras permette di creare layers completamente tuoi.
Questo è fondamentale quando: la logica di business è particolare, il preprocessing deve stare dentro il modello, vuoi fare ricerca, vuoi architetture non standard, vuoi integrare logiche ERP direttamente nella rete.
Un layer di keras è:
- un oggetto con uno stato
- che può avere pesi
- che trasforma tensori in altri tensori
input tensor -> trasformazione -> output tensor
Un layer può:
- avere pesi traibabili
- NON avere pesi
- contenere logica matematica arbitraria

Keras internamente usa OOP (object oriented programmar)
Quindi quando crei un custom layer stai creando una classe Python


Architettura su Misura
Perchè i layer predefiniti non sempre bastano
Sebbene Keras offra una vasta libreria di layer pronti all'uso, la ricerca e lo sviluppo di modelli all'avanguardia richiedono spesso logiche di calcolo uniche. Immaginate di voler implementare una variante della convoluzione e una funzione di attivazione parametrizzata che non esiste nel catalogo standard.

TensorFlow permette di definire nuovi mattoni computazionali in modo modulare, garantendo che siano compatibili con il sistema di calcolo del gradiente automatico e con le pipeline di salvataggio dei modelli.
Creare un custom layer significa avere il pieno controllo sul calcolo pur mantenendo la potenza di TensorFlow per la gestione della memorie e dei gradienti.

Ereditarietà e Struttura

Fondamenta della classe 'layer'
Tutto parte da un concetto fondamentale all'interno della programmazione: l'ereditarietà.
Il nosto nuovo layer deve essere figlio della classe base layer. E' come se TensorFlow ci desse lo scheletro di una creatura e noi dobbiamo decidere come farlo muovere e mangiare. Ereditando otteniamo gratis tutta l'infrastruttura per trovare i pesi e calcolare le derivate. Questo trasforma una formula matematica complessa in un oggetto semplice che può essere inserito in un modello 'sequential' esattamente come fosse un layer nativo.
- Ogni layer personalizzato deve ereditare dalla classe madre tf.kers.layers.Layer. Questo fornisce l'infrastruttura necessaria per la gestione dei pesi e la tracciabilità delle operazioni.
- L'incapsulamento permette di trattare una formula complessa come un singolo oggetto atomico, migliorando la leggibilità e la  manutenibilità del codice del modello.
- Un layer personalizzato si comporta esattamente come un layer stadard: può essere inserito in modelli Sequantial o utilizzati tramite le API funzionali.
- La classe base gestisce automaticamente la registrazione dei parametri addestrabili e la logica di inferenza.

Il Costruttore
- Configurazione iniziale: Nel metodo __init__ definiamo gli iperparametri del layer (caratteristiche esterne del layer, che non cambiano durante l'addestramento), come il numero di unità o eventuali parametri di regolarizzazione, ma non ancora i pesi addestrabili,
- Inizializzazione dello Stato: E' qui che chiamiamo super().__init__() per assicurarsi che Keras possa tracciare correttamente la nostra istanza all'interno del grafo computazionale. Diciamo a Keras: 'guarda che questo è un pezzo ufficiale del tuo grafo, permettendo al sistema di registrarsi correttamente.
- Flessibilità Statica: tutto ciò che non dipende dalla forma dei dati in ingresso (input shape) può essere configurato in questa fase iniziale dello sviluppo del layer.
Dobbiamo ora capire come il layer si adatti ai dati che riceve

Modularità ed Estensibilità
Il layer come funzione parametrizzata.
Crea un layer personalizzato significa definire una funzione matematica dove alcuni parametri sono fissi (iperparametri) e altri vengono appresi durante il processo di training (pesi).
Separando la logica della trasformazione, dalla gestione dei parametri, lasciamo che tensorflow faccia il lavoro pesante di ottimizzazione. 
Questa separazione è fondamentale per permettere al programmatore di concentrarsi sulla logica della trasformazione, lasciando a TensorFlow il compito di ottimizzare le prestazioni.
Distinzione netta tra iperparametri e pesi, gli iperparametri sono come le impostazioni di una ricetta, i pesi invece sono gli ingredienti che si adattano durante la cottura (ovvero addestramento) per ottenere il sapore perfetto.
Come fa il layer a sapere quando spazio occupare in memoria se non conosce ancora l'input?

Il ciclo di Vita: Build e Call
Creazione dello stato e calcolo del forward pass.
Qui entra in gioco il concetto di stato ritardato, TensorFlow è inteligente, aspetta di vedere il primo barch di dati prima di creare effettivamente i pesi del layer. Questo avviene tramite due metodi: 'buil' e 'call'. Build è come un sarto che prende le misure per un vestito, non può iniziare a cucire finchè non sa quanto è alto un cliente. Call invece è il ciclo in azione, il momento in cui i dati vengono effettivamente indossata e trasformati..
Per rendere un layer realmente flessibile, TenrsorFlow introduce una separazione tra il momento di cui definiamo il layer e il modello in sui conosciamo la dimensione dei dati che processerà.
Implementare correttamente 'build' e 'call' è il segreto per creare blocchi che possono adattarsi automaticamente a diversi input senza richiedere modifiche manuali al codice.

Gestione Dinamica dei Pesi
Il ruolo cruciale di 'build'
- Il metodo 'build' viene invocato automaticamente la prima volta che il layer riceve dei dati. Serve per creare i pesi la cui forma dipende dalla dimensione dell'input. Qui avviene la creazioe fisica dello stato.
- Utilizziamo 'add_weight' per registrare variabili addestrabili, definendo inizializzatori e vincoli che TensorFlow userà durante l'ottimizzazione
- Il metodo 'call' contiene la logica matematica vera e propria. Riceve i tenrsoflow in input e restituisce i tensori traformati applicando i pesi creati. Qui riceve la scenza del layer personalizzato
- Lo stato del layer viene congelato dopo la prima chiamata, garantendo coerenza tra i vari batch di dati.

Esecuzione e Autograph
Come fa questo codice personalizzato ed essere così veloce durante l'addestramento?
TensorFlow non esegue il codice Python riga per riga per ogni singolo dati, sarebbe troppo lento, usa un sistema chiamato Autograph per convertire il nostro metodo call in un grafico statico e super efficiente.
Per questo motivo il metodo call deve essere scritto in puro linguaggio TernsoFlow, ricevendo, come vantaggio, che non dobbiamo scrivere una sola riga di codice per la back-propagation.

Tracciamento dei parametri
Pesi addestrabili contro non addestrabili
Attraverso 'add_weight', possiamo decidere se un parametro deve essere aggiornato dal Gradient Descent o se deve essere un valore costante (ignorato completamente dal Gradient Descent, non addestrabili) o calcolato diversamente.
Questo controllo granulare  è ciò che permette di implementare tecniche avanza come la 'batch Normalization' o layer di memoria personalizzati che mantengono statistiche dei dati precedenti.

PRATICA: Trasformazioni Custom
Sviluppo di un layer matematico.
Metteremo in pratica la teoria creando un layer che applica una trasformazione lineare scalata. Impareremo come definire coefficienti addestrabili che non rientrano negli schemi stanrdad dei layer densi.
Vedremo come manimpolare i tensori direttametne e come integrare il nostro nuovo blocco all'interno di un modello Keras funzionante.

Implementazione Logica
Moltiplicazione e Addizione Custom
- Il layer riceverà in input e applicherà una formula dove ogni elemento è moltiplicato per un peso scalare e addizionato a un bias.
- Useremo il metodo 'build' per assicurarci che i vettori dei pesi abbiano la stessa dimensione dell'input che riceveremo
- In 'call' TensorFlow usaerà il 'broadcasting' applicherà i nostri pesi ad ogni singola riga del dataset senza che si debba scrivere un loop ingegnoso.
- La formula implementata rappresenterà una versione generalizzata della trasformazione lineare punto a punto Z=X*w+b

Verifica del Modello
La prova del nove è sempre model.summary
Se abbiamo fatto tutto bene vedremo il nostro layer personalizzato con il conteggio esatto dei parametri addestrabili che abbiao definito in fase di build. Dovremmo anche addestrare il layer con dati fittizi per vedere se i dati in uscita hanno senso. E' il debugging tipico di chi costruisce architetture da zero.

Oltre il Caclolo Lineare
Potenzialità delle trasformazioni custom
Sebbene l'esempio sia semplice, la stessa logica permette di inserire operazioni di Fourier, filtri di elaborazione segnali o logiche di attenzione personalizzate per il linguaggi naturale.
Essere in grado di scrivere i propri layer è la competenza che distingue un utilizzatore di librerie da un ricervatore di Deep Learning

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

#1.DEFINIZIONE DEL LAYER PERSONALIZZATO
#Ereditando della classe base 'Layer' per ottenre tutte le funzionalità di un layer standard di TensorFlow
class CustomLayer(layers.Layer):
    def __init__(self, units=32,**kwargs): #32=numero di neuroni che andremmo a creare in questo layer
        #Inizializzazione: definiamo gli iperparametri del layer (es. n. di neuroni)
        #Chiamiamo super() per permettere a Keras di gestire correttamente il layer
        super(CustomLayer, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        """
        Definiamo w e b, gli si aggiungono dei pesi, che dovreanno gestire una shape, questi parametri sono aggiornabili
        
        Il metodo build viene eseguito automaticamente la prima volta che il layer viene chiamato in input
        Serve per creare i pesi solo quando conosciamo la dimensione dell'input (input_shape)        """
        #Creazione dei pesi del layer
        #Creazione del 'kernel' (matrice dei pesi):
        #La forma è (numero_caratteristiche_input,unita_output)
        self.w=self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=True, #indica a TensorFlow di calcolare i gradienti per questo peso
            name='Kernel') #nome del peso, utile per il debug e la visualizzazione dei pesi del modello
        #Creazione del 'Bias' (vettore di offset):
        #Uno scalare per ogni unità di output
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True,
            name='Bias'
        )

    def call(self, inputs):
        """ 
        Il metodo call contiene la logica del 'Forwar Pass'.
        Qui definiamo l'operazione matematica da eseguire sui dati in ingresso (input), utilizzando i pesi (w) e il bias (b) definito nel metodo build.
        """
        #Definizione della logica di calcolo del layer
        return tf.matmul(inputs, self.w) + self.b #semplice operazione di matrice moltiplicata per i pesi più il bias
        #la classica y=X*W+B, dove X è l'input, W sono i pesi e B è il bias


# ---- FASE  DI VERIFICA (test) DEL LAYER PERSONALIZZATO

#Creiamo un tensore di esempio (batch_size=1, features=3)
input_data=tf.constant([[1.0,2.0,3.0]],dtype=tf.float32)

#Istanziamo il layer con 4 unità di output
custom_layer = CustomLayer(units=4)

#Applichiamo il layer ai dati di input (questo invocherà internameto build e call)
output=custom_layer(input_data)

#Visualizzazione dei risultati per verificare la trasformazione dei dati attraverso il layer personalizato
print("Forma dell'input:",input_data.shape)
print("Forma dell'output:",output.shape)
print("Risultato dell'output:",output.numpy())


# ---- INTEGRAZIONE DEL LAYER PERSONALIZZATO IN UN MODELLO KERAS


#2.CREAZIONE DEL MODELLO UTILIZZANDO IL LAYER PERSONALIZZATO

#Dimostriamo che il layer personalizzato sia perfettamente compatibile con gli
#altri layer standard di Keras (Dense,Activation, ecc.) e che possa essere utilizzato 
#in un modello sequenziale standard di Keras.


#definiamo il nostro layer in un modello, un semplice sequential con il layer personalizzato 
#seguito da layer standard di Kersas
model = models.Sequential([
    layers.Input(shape=(3 ,)),  #Definiamo la dimensione dell'input (3 caratteristiche in ingresso)
    CustomLayer(units=8),       #Layer personalizzato con 8 unità (8 neuroni)
    layers.Activation('relu'),  #Funzione di attivazione ReLU (layer standard di Keras)
    layers.Dense(1)             #Output layer con 1 unità, layer finale per la regressione (o classificazione binaria)
])

#Mostriamo l'architettura: noteremo i pesi del nostro layer nel conteggio dei parametri del modello
model.summary()

#3.COMPILAZIONE DEL MODELLO
model.compile(optimizer='adam', loss='mse')
#4.GENERAZIONE DI DATI DI ESEMPIO
X_train = np.random.rand(100, 3)  # 100 campioni, 3 caratteristiche ciascuno
y_train = np.random.rand(100, 1)   # 100 campioni, 1 target ciascuno
#5.ADDENDA DEL MODELLO
model.fit(X_train, y_train, epochs=5)
#6.PREDIZIONE CON IL MODELLO
X_test = np.random.rand(10, 3)  # 10 campioni di test
predictions = model.predict(X_test)
print(predictions)




Forma dell'input: (1, 3)
Forma dell'output: (1, 4)
Risultato dell'output: [[ 0.03056335  0.08240427 -0.2534642   0.03093162]]


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ custom_layer_8 (CustomLayer)    │ (None, 8)              │            32 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41 (164.00 B)

 Trainable params: 41 (164.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.3148
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.2954
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2761
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.2561
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.2353
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
[[0.19288439]
 [0.2096712 ]
 [0.07849142]
 [0.17670274]
 [0.19664443]
 [0.157864  ]
 [0.15735152]
 [0.12250692]
 [0.13449469]
 [0.16846028]]


L'ereditarietà permette di estendere Keras mentre i metodi build e call governano il ciclo di vita del nostro codice.
Build per le forme (per lostato ritardato)
Call per la matematica (per la logica computazionale del forware pass)